In [1]:
import sys
from pathlib import Path
from time import perf_counter

import joblib
import pandas as pd
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)

ROOT = Path(r"C:\tcc\network-ids-ml-generalization")
DATA_DIR = ROOT / "datasets" / "final" / "ml"
BASELINE_MODEL_DIR = ROOT / "models" / "baseline"
RANDOMIZED_MODEL_DIR = ROOT / "models" / "randomized_search"
TRAINING_RESULTS_DIR = ROOT / "results" / "model_training"
OUTPUT_DIR = ROOT / "results" / "model_evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT))
from src.utils.features import split_features

reference_df = pd.read_parquet(DATA_DIR / "cicids2017_train.parquet")
numerical_features, categorical_features = split_features(reference_df)
feature_columns = numerical_features + categorical_features
del reference_df


TEST_CONFIGS = {
    "cicids2017": ["cicids2017", "iot23", "unsw_nb15"],
    "unsw_nb15": ["unsw_nb15", "cicids2017", "iot23"],
    "iot23": ["iot23", "cicids2017", "unsw_nb15"],
    "cicids2017_unsw_nb15": ["iot23"],
    "cicids2017_iot23": ["unsw_nb15"],
    "unsw_nb15_iot23": ["cicids2017"],
}

TEST_DATASETS = ["cicids2017", "iot23", "unsw_nb15"]
SINGLE_DATASET_TRAINING = set(TEST_DATASETS)
MODELS_TO_EVALUATE = [
    "decision_tree",
    "random_forest",
    "xgboost",
    "linear_svm",
    "mlp",
]

MODEL_VERSIONS = {
    "baseline": {
        "label": "padrão",
        "model_dir": BASELINE_MODEL_DIR,
    },
    "randomized_search": {
        "label": "RandomizedSearchCV",
        "model_dir": RANDOMIZED_MODEL_DIR,
    },
}

In [2]:
CV_METRICS = [
    "precision_attack",
    "recall_attack",
    "f1_attack",
    "precision_benign",
    "recall_benign",
    "f1_benign",
    "f1_macro",
    "accuracy",
    "balanced_accuracy",
]


def load_training_summaries(training_method):
    frames = []
    method_dir = TRAINING_RESULTS_DIR / training_method

    for model_name in MODELS_TO_EVALUATE:
        summary_path = method_dir / model_name / "summary.csv"
        if not summary_path.exists():
            raise FileNotFoundError(
                f"Resumo de treinamento não encontrado: {summary_path}. "
                "Execute primeiro os notebooks 07 e 08."
            )
        frame = pd.read_csv(summary_path)
        frame["training_method"] = training_method
        frames.append(frame)

    return pd.concat(frames, ignore_index=True).set_index(
        ["training_method", "model", "training_source"]
    )


def scenario_type(training_name, test_name):
    if training_name == test_name:
        return "intra_dataset"
    if training_name in SINGLE_DATASET_TRAINING:
        return "cross_dataset"
    return "combined_to_unseen"


def training_metadata(summary_row):
    metadata = {
        "training_rows": summary_row.get("training_rows"),
        "params": summary_row.get("params", "{}"),
        "cv_n_splits": summary_row.get("n_splits"),
        "random_state": summary_row.get("random_state"),
        "training_elapsed_seconds": summary_row.get("elapsed_seconds"),
        "training_elapsed_minutes": summary_row.get("elapsed_minutes"),
        "final_fit_time_seconds": summary_row.get("refit_time_seconds"),
        "model_size_mb": summary_row.get("model_size_mb"),
    }

    for metric in CV_METRICS:
        metadata[f"cv_{metric}_mean"] = summary_row.get(f"{metric}_mean")
        metadata[f"cv_{metric}_std"] = summary_row.get(f"{metric}_std")

    return metadata

In [3]:
columns_to_load = feature_columns + ["label_binary"]
test_dataframes = {
    dataset_name: pd.read_parquet(
        DATA_DIR / f"{dataset_name}_test.parquet",
        columns=columns_to_load,
    )
    for dataset_name in TEST_DATASETS
}

training_summaries = pd.concat(
    [
        load_training_summaries("baseline"),
        load_training_summaries("randomized_search"),
    ]
)

evaluation_results = []

for training_name, test_names in TEST_CONFIGS.items():
    for model_name in MODELS_TO_EVALUATE:
        report_dir = OUTPUT_DIR / training_name / model_name
        report_dir.mkdir(parents=True, exist_ok=True)

        for version, version_config in MODEL_VERSIONS.items():
            model_path = (
                version_config["model_dir"]
                / model_name
                / f"{training_name}.joblib"
            )
            pipeline = joblib.load(model_path)
            summary_row = training_summaries.loc[
                (version, model_name, training_name)
            ]

            for test_name in test_names:
                test_df = test_dataframes[test_name]
                X_test = test_df[feature_columns]
                y_test = test_df["label_binary"].eq("ATTACK").astype("int8")

                prediction_start = perf_counter()
                predictions = pipeline.predict(X_test)
                prediction_time_seconds = perf_counter() - prediction_start

                metrics_start = perf_counter()
                report_values = classification_report(
                    y_test,
                    predictions,
                    labels=[0, 1],
                    target_names=["BENIGN", "ATTACK"],
                    output_dict=True,
                    zero_division=0,
                )
                report_text = classification_report(
                    y_test,
                    predictions,
                    labels=[0, 1],
                    target_names=["BENIGN", "ATTACK"],
                    digits=4,
                    zero_division=0,
                )
                matrix_values = confusion_matrix(y_test, predictions, labels=[0, 1])
                balanced_accuracy = balanced_accuracy_score(y_test, predictions)
                metrics_calculation_time_seconds = perf_counter() - metrics_start

                matrix = pd.DataFrame(
                    matrix_values,
                    index=["Real BENIGN", "Real ATTACK"],
                    columns=["Predito BENIGN", "Predito ATTACK"],
                )
                true_negative, false_positive, false_negative, true_positive = (
                    matrix_values.ravel()
                )

                title = (
                    f"Treino: {training_name} | Modelo: {model_name} | "
                    f"Configuração: {version_config['label']} | Teste: {test_name}"
                )
                report_path = report_dir / f"{version}_{test_name}.txt"
                matrix_path = report_dir / f"{version}_{test_name}_confusion_matrix.txt"
                report_path.write_text(f"{title}\n\n{report_text}", encoding="utf-8")
                matrix_path.write_text(f"{title}\n\n{matrix}", encoding="utf-8")

                result = {
                    "model": model_name,
                    "training_method": version,
                    "training_source": training_name,
                    "test_source": test_name,
                    "scenario_type": scenario_type(training_name, test_name),
                    "test_rows": len(test_df),
                    "benign_support": int(report_values["BENIGN"]["support"]),
                    "attack_support": int(report_values["ATTACK"]["support"]),
                    "accuracy": report_values["accuracy"],
                    "balanced_accuracy": balanced_accuracy,
                    "precision_benign": report_values["BENIGN"]["precision"],
                    "recall_benign": report_values["BENIGN"]["recall"],
                    "f1_benign": report_values["BENIGN"]["f1-score"],
                    "precision_attack": report_values["ATTACK"]["precision"],
                    "recall_attack": report_values["ATTACK"]["recall"],
                    "f1_attack": report_values["ATTACK"]["f1-score"],
                    "f1_macro": report_values["macro avg"]["f1-score"],
                    "f1_weighted": report_values["weighted avg"]["f1-score"],
                    "true_negative": int(true_negative),
                    "false_positive": int(false_positive),
                    "false_negative": int(false_negative),
                    "true_positive": int(true_positive),
                    "prediction_time_seconds": prediction_time_seconds,
                    "prediction_rows_per_second": (
                        len(test_df) / prediction_time_seconds
                        if prediction_time_seconds > 0
                        else float("inf")
                    ),
                    "metrics_calculation_time_seconds": metrics_calculation_time_seconds,
                    "model_file": str(model_path.relative_to(ROOT)),
                    "classification_report_file": str(report_path.relative_to(ROOT)),
                    "confusion_matrix_file": str(matrix_path.relative_to(ROOT)),
                }
                result.update(training_metadata(summary_row))
                evaluation_results.append(result)

evaluation_df = pd.DataFrame(evaluation_results).sort_values(
    ["training_source", "test_source", "model", "training_method"]
)
evaluation_df.to_csv(OUTPUT_DIR / "evaluation_summary.csv", index=False)
display(evaluation_df)

,model,training_method,training_source,test_source,scenario_type,test_rows,benign_support,attack_support,accuracy,balanced_accuracy,...,cv_recall_benign_mean,cv_recall_benign_std,cv_f1_benign_mean,cv_f1_benign_std,cv_f1_macro_mean,cv_f1_macro_std,cv_accuracy_mean,cv_accuracy_std,cv_balanced_accuracy_mean,cv_balanced_accuracy_std
0,decision_tree,baseline,cicids2017,cicids2017,intra_dataset,100925,80740,20185,0.994055,0.992123,...,0.995377,0.000327,0.996133,0.000145,0.990361,0.000363,0.993817,0.000232,0.991478,0.000652
3,decision_tree,randomized_search,cicids2017,cicids2017,intra_dataset,100925,80740,20185,0.993579,0.992104,...,0.994846,0.000457,0.996057,0.000204,0.990190,0.000499,0.993698,0.000324,0.991977,0.000383
18,linear_svm,baseline,cicids2017,cicids2017,intra_dataset,100925,80740,20185,0.888135,0.763197,...,0.971527,0.000929,0.932627,0.000560,0.797837,0.001565,0.887706,0.000912,0.761975,0.001436
21,linear_svm,randomized_search,cicids2017,cicids2017,intra_dataset,100925,80740,20185,0.931504,0.942829,...,0.924174,0.001156,0.955983,0.000880,0.902885,0.001845,0.931916,0.001341,0.943530,0.001734
24,mlp,baseline,cicids2017,cicids2017,intra_dataset,100925,80740,20185,0.993678,0.992909,...,0.994427,0.000688,0.995521,0.000497,0.988847,0.001267,0.992840,0.000799,0.990461,0.002442
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,mlp,randomized_search,unsw_nb15_iot23,cicids2017,combined_to_unseen,100925,80740,20185,0.790706,0.703233,...,0.991884,0.000453,0.994713,0.000215,0.986930,0.000540,0.991564,0.000345,0.991085,0.001020
112,random_forest,baseline,unsw_nb15_iot23,cicids2017,combined_to_unseen,100925,80740,20185,0.715799,0.486611,...,0.993028,0.000160,0.995202,0.000092,0.988108,0.000225,0.992339,0.000146,0.991307,0.000218
113,random_forest,randomized_search,unsw_nb15_iot23,cicids2017,combined_to_unseen,100925,80740,20185,0.711826,0.483311,...,0.992545,0.000184,0.995611,0.000119,0.989162,0.000292,0.993000,0.000189,0.993682,0.000287
114,xgboost,baseline,unsw_nb15_iot23,cicids2017,combined_to_unseen,100925,80740,20185,0.749547,0.549560,...,0.992489,0.000188,0.995595,0.000166,0.989124,0.000410,0.992974,0.000264,0.993702,0.000441


In [4]:
comparison_keys = [
    "model",
    "training_source",
    "test_source",
    "scenario_type",
]
comparison_metrics = [
    "accuracy",
    "balanced_accuracy",
    "precision_attack",
    "recall_attack",
    "f1_attack",
    "f1_macro",
    "prediction_time_seconds",
    "cv_f1_attack_mean",
    "cv_f1_attack_std",
    "cv_balanced_accuracy_mean",
    "cv_balanced_accuracy_std",
    "training_elapsed_seconds",
]

baseline_comparison = evaluation_df.loc[
    evaluation_df["training_method"].eq("baseline"),
    comparison_keys + comparison_metrics,
].rename(columns={metric: f"{metric}_baseline" for metric in comparison_metrics})

randomized_comparison = evaluation_df.loc[
    evaluation_df["training_method"].eq("randomized_search"),
    comparison_keys + comparison_metrics + ["params"],
].rename(
    columns={
        **{metric: f"{metric}_randomized_search" for metric in comparison_metrics},
        "params": "params_randomized_search",
    }
)

comparison_df = baseline_comparison.merge(
    randomized_comparison,
    on=comparison_keys,
    how="inner",
)
for metric in comparison_metrics:
    comparison_df[f"{metric}_delta"] = (
        comparison_df[f"{metric}_randomized_search"]
        - comparison_df[f"{metric}_baseline"]
    )

comparison_df.to_csv(
    OUTPUT_DIR / "baseline_vs_randomized_search.csv",
    index=False,
)
display(comparison_df)

,model,training_source,test_source,scenario_type,accuracy_baseline,balanced_accuracy_baseline,precision_attack_baseline,recall_attack_baseline,f1_attack_baseline,f1_macro_baseline,...,precision_attack_delta,recall_attack_delta,f1_attack_delta,f1_macro_delta,prediction_time_seconds_delta,cv_f1_attack_mean_delta,cv_f1_attack_std_delta,cv_balanced_accuracy_mean_delta,cv_balanced_accuracy_std_delta,training_elapsed_seconds_delta
0,decision_tree,cicids2017,cicids2017,intra_dataset,0.994055,0.992123,0.981512,0.988903,0.985193,0.990737,...,-0.003015,0.000743,-0.001154,-0.000727,-0.001975,-0.000267,0.000211,0.000499,-0.000270,13.265274
1,linear_svm,cicids2017,cicids2017,intra_dataset,0.888135,0.763197,0.829225,0.554967,0.664926,0.798893,...,-0.069519,0.406738,0.183928,0.103393,-0.004761,0.186739,0.000202,0.181555,0.000298,40.298118
2,mlp,cicids2017,cicids2017,intra_dataset,0.993678,0.992909,0.977105,0.991627,0.984313,0.990177,...,-0.000532,0.001734,0.000583,0.000362,0.006075,0.003075,-0.000736,0.003328,-0.001909,146.944374
3,random_forest,cicids2017,cicids2017,intra_dataset,0.994848,0.994309,0.981066,0.993411,0.987200,0.991987,...,-0.000191,0.000050,-0.000072,-0.000045,0.600645,0.000227,0.000096,0.000279,-0.000062,572.989819
4,xgboost,cicids2017,cicids2017,intra_dataset,0.994580,0.994346,0.979256,0.993956,0.986551,0.991579,...,0.000280,-0.000347,-0.000029,-0.000017,0.194159,0.000136,0.000136,0.000058,0.000114,46.052998
5,decision_tree,cicids2017,iot23,cross_dataset,0.804865,0.523105,0.647094,0.053505,0.098838,0.494712,...,-0.631258,0.003418,-0.074059,-0.396767,-0.000812,-0.000267,0.000211,0.000499,-0.000270,13.265274
6,linear_svm,cicids2017,iot23,cross_dataset,0.110964,0.107809,0.028094,0.102551,0.044105,0.106593,...,0.190306,0.893584,0.314148,0.170547,-0.000986,0.186739,0.000202,0.181555,0.000298,40.298118
7,mlp,cicids2017,iot23,cross_dataset,0.251117,0.460887,0.185664,0.810503,0.302121,0.247096,...,-0.000138,-0.000297,-0.000203,-0.000485,-0.015838,0.003075,-0.000736,0.003328,-0.001909,146.944374
8,random_forest,cicids2017,iot23,cross_dataset,0.099846,0.064671,0.001721,0.006044,0.002679,0.091220,...,0.000026,0.000099,0.000041,-0.000792,0.326585,0.000227,0.000096,0.000279,-0.000062,572.989819
9,xgboost,cicids2017,iot23,cross_dataset,0.094922,0.063228,0.002934,0.010404,0.004577,0.087405,...,0.070864,-0.005995,0.003744,0.357981,0.237286,0.000136,0.000136,0.000058,0.000114,46.052998


In [5]:
def show_reports(
    training_name,
    test_name,
    show_baseline=True,
    show_randomized_search=False,
    show_classification_report=True,
    show_confusion_matrix=True,
):
    versions = []
    if show_baseline:
        versions.append("baseline")
    if show_randomized_search:
        versions.append("randomized_search")

    for model_name in MODELS_TO_EVALUATE:
        report_dir = OUTPUT_DIR / training_name / model_name

        for version in versions:
            if show_classification_report:
                report_path = report_dir / f"{version}_{test_name}.txt"
                print(report_path.read_text(encoding="utf-8"))

            if show_confusion_matrix:
                matrix_path = report_dir / f"{version}_{test_name}_confusion_matrix.txt"
                print(matrix_path.read_text(encoding="utf-8"))

In [6]:
show_reports("cicids2017", "cicids2017")

Treino: cicids2017 | Modelo: decision_tree | Configuração: padrão | Teste: cicids2017

              precision    recall  f1-score   support

      BENIGN     0.9972    0.9953    0.9963     80740
      ATTACK     0.9815    0.9889    0.9852     20185

    accuracy                         0.9941    100925
   macro avg     0.9894    0.9921    0.9907    100925
weighted avg     0.9941    0.9941    0.9941    100925

Treino: cicids2017 | Modelo: decision_tree | Configuração: padrão | Teste: cicids2017

             Predito BENIGN  Predito ATTACK
Real BENIGN           80364             376
Real ATTACK             224           19961
Treino: cicids2017 | Modelo: random_forest | Configuração: padrão | Teste: cicids2017

              precision    recall  f1-score   support

      BENIGN     0.9983    0.9952    0.9968     80740
      ATTACK     0.9811    0.9934    0.9872     20185

    accuracy                         0.9948    100925
   macro avg     0.9897    0.9943    0.9920    100925
weighted

In [7]:
show_reports("cicids2017", "unsw_nb15")

Treino: cicids2017 | Modelo: decision_tree | Configuração: padrão | Teste: unsw_nb15

              precision    recall  f1-score   support

      BENIGN     0.7957    0.9708    0.8746     80740
      ATTACK     0.0244    0.0029    0.0052     20185

    accuracy                         0.7772    100925
   macro avg     0.4101    0.4869    0.4399    100925
weighted avg     0.6414    0.7772    0.7007    100925

Treino: cicids2017 | Modelo: decision_tree | Configuração: padrão | Teste: unsw_nb15

             Predito BENIGN  Predito ATTACK
Real BENIGN           78382            2358
Real ATTACK           20126              59
Treino: cicids2017 | Modelo: random_forest | Configuração: padrão | Teste: unsw_nb15

              precision    recall  f1-score   support

      BENIGN     0.8000    0.9998    0.8888     80740
      ATTACK     0.0000    0.0000    0.0000     20185

    accuracy                         0.7999    100925
   macro avg     0.4000    0.4999    0.4444    100925
weighted av

In [8]:
show_reports(
    "cicids2017_iot23",
    "unsw_nb15",
    show_randomized_search=True,
)

Treino: cicids2017_iot23 | Modelo: decision_tree | Configuração: padrão | Teste: unsw_nb15

              precision    recall  f1-score   support

      BENIGN     0.8101    0.9934    0.8925     80740
      ATTACK     0.7231    0.0686    0.1253     20185

    accuracy                         0.8085    100925
   macro avg     0.7666    0.5310    0.5089    100925
weighted avg     0.7927    0.8085    0.7390    100925

Treino: cicids2017_iot23 | Modelo: decision_tree | Configuração: padrão | Teste: unsw_nb15

             Predito BENIGN  Predito ATTACK
Real BENIGN           80210             530
Real ATTACK           18801            1384
Treino: cicids2017_iot23 | Modelo: decision_tree | Configuração: RandomizedSearchCV | Teste: unsw_nb15

              precision    recall  f1-score   support

      BENIGN     0.8061    0.9937    0.8901     80740
      ATTACK     0.6340    0.0439    0.0822     20185

    accuracy                         0.8037    100925
   macro avg     0.7201    0.5188  

In [9]:
show_reports("unsw_nb15", "unsw_nb15")

Treino: unsw_nb15 | Modelo: decision_tree | Configuração: padrão | Teste: unsw_nb15

              precision    recall  f1-score   support

      BENIGN     0.9892    0.9877    0.9884     80740
      ATTACK     0.9512    0.9568    0.9540     20185

    accuracy                         0.9815    100925
   macro avg     0.9702    0.9722    0.9712    100925
weighted avg     0.9816    0.9815    0.9816    100925

Treino: unsw_nb15 | Modelo: decision_tree | Configuração: padrão | Teste: unsw_nb15

             Predito BENIGN  Predito ATTACK
Real BENIGN           79749             991
Real ATTACK             873           19312
Treino: unsw_nb15 | Modelo: random_forest | Configuração: padrão | Teste: unsw_nb15

              precision    recall  f1-score   support

      BENIGN     0.9969    0.9858    0.9913     80740
      ATTACK     0.9456    0.9878    0.9662     20185

    accuracy                         0.9862    100925
   macro avg     0.9712    0.9868    0.9788    100925
weighted avg  

In [10]:
show_reports("iot23", "iot23")

Treino: iot23 | Modelo: decision_tree | Configuração: padrão | Teste: iot23

              precision    recall  f1-score   support

      BENIGN     0.9987    0.9997    0.9992     80740
      ATTACK     0.9987    0.9947    0.9967     20185

    accuracy                         0.9987    100925
   macro avg     0.9987    0.9972    0.9979    100925
weighted avg     0.9987    0.9987    0.9987    100925

Treino: iot23 | Modelo: decision_tree | Configuração: padrão | Teste: iot23

             Predito BENIGN  Predito ATTACK
Real BENIGN           80713              27
Real ATTACK             106           20079
Treino: iot23 | Modelo: random_forest | Configuração: padrão | Teste: iot23

              precision    recall  f1-score   support

      BENIGN     0.9987    0.9997    0.9992     80740
      ATTACK     0.9986    0.9947    0.9966     20185

    accuracy                         0.9987    100925
   macro avg     0.9986    0.9972    0.9979    100925
weighted avg     0.9987    0.9987    0